# ESPN Public API - Weekly Scheduled Update

This notebook automatically fetches the **latest week's NFL data** from ESPN's public API.

**Schedule:** Runs weekly on Tuesdays at 8:00 AM (America/Chicago)

**What it does:**
- Detects the current NFL season and latest completed week
- Fetches complete box scores for all games in that week
- Stores data in `main.fantasai.bronze_weekly_stats` and `main.fantasai.silver_weekly_stats`
- Includes 10 stat categories: passing, rushing, receiving, defensive, kicking, punting, returns, fumbles, interceptions

**Source:** ESPN Public API (no authentication required)
**Data Format:** JSON stats stored per player per week

In [0]:
import requests
import json
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
import time
from datetime import datetime

# ESPN Public API Configuration - NO API KEY NEEDED!
BASE_URL = "https://site.api.espn.com/apis/site/v2/sports/football/nfl"

print("✓ Libraries imported")
print(f"✓ ESPN Public API configured: {BASE_URL}")
print(f"✓ Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [0]:
# Fetch only the latest week's data
# Auto-detects current season and most recent completed week

print("="*70)
print("ESPN PUBLIC API - LATEST DATA FETCH")
print("="*70)
print(f"\nCurrent date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# Determine current season and week
# NFL regular season typically runs September-January (Weeks 1-18)
current_date = datetime.now()
current_year = current_date.year
current_month = current_date.month

# Determine season based on current date
if current_month >= 9:  # September onwards = current year's season
    CURRENT_SEASON = current_year
else:  # January-August = previous year's season
    CURRENT_SEASON = current_year - 1

SEASON_TYPE = 2  # Regular season

print(f"Detected season: {CURRENT_SEASON}")
print("Finding latest week with games...\n")

# Try to find the most recent week with completed games (check backwards from week 18)
latest_week = None
for check_week in range(18, 0, -1):
    try:
        response = requests.get(
            f"{BASE_URL}/scoreboard",
            params={
                "week": check_week,
                "seasontype": SEASON_TYPE,
                "year": CURRENT_SEASON
            },
            timeout=30
        )
        
        if response.status_code == 200:
            scoreboard_data = response.json()
            events = scoreboard_data.get('events', [])
            
            if events:
                # Check if games have been played (not just scheduled)
                completed_games = sum(1 for e in events if e.get('status', {}).get('type', {}).get('completed', False))
                
                if completed_games > 0:
                    latest_week = check_week
                    print(f"✓ Found latest week: {latest_week} ({completed_games}/{len(events)} games completed)\n")
                    break
    except:
        continue

if not latest_week:
    print("⚠️  No completed games found - season may not have started yet")
    print("Defaulting to Week 1\n")
    latest_week = 1

# Fetch data for the latest week
WEEK = latest_week
SEASON = CURRENT_SEASON

print(f"Fetching data for Season {SEASON}, Week {WEEK}...\n")

try:
    # Fetch scoreboard
    response = requests.get(
        f"{BASE_URL}/scoreboard",
        params={
            "week": WEEK,
            "seasontype": SEASON_TYPE,
            "year": SEASON
        },
        timeout=30
    )
    
    if response.status_code == 200:
        scoreboard_data = response.json()
        events = scoreboard_data.get('events', [])
        
        print(f"✓ Found {len(events)} games\n")
        
        # Fetch box scores for each game
        all_player_records = []
        
        for idx, event in enumerate(events, 1):
            game_id = event.get('id')
            game_name = event.get('shortName', 'Unknown')
            
            print(f"[{idx}/{len(events)}] {game_name}...", end=' ')
            
            try:
                summary_response = requests.get(
                    f"{BASE_URL}/summary",
                    params={"event": game_id},
                    timeout=30
                )
                
                if summary_response.status_code == 200:
                    summary_data = summary_response.json()
                    boxscore = summary_data.get('boxscore', {})
                    players = boxscore.get('players', [])
                    
                    game_player_count = 0
                    
                    # Process players
                    for team_data in players:
                        team_name = team_data.get('team', {}).get('abbreviation', 'UNK')
                        stat_groups = team_data.get('statistics', [])
                        
                        for stat_group in stat_groups:
                            stat_category = stat_group.get('name')
                            athletes = stat_group.get('athletes', [])
                            stat_labels = stat_group.get('labels', [])
                            
                            for athlete in athletes:
                                player_info = athlete.get('athlete', {})
                                player_id = player_info.get('id')
                                player_name = player_info.get('displayName')
                                position = player_info.get('position', {}).get('abbreviation', 'N/A')
                                stats = athlete.get('stats', [])
                                
                                player_stats = {
                                    'player_name': player_name,
                                    'team': team_name,
                                    'position': position,
                                    'game_id': game_id,
                                    'game_name': game_name,
                                    'stat_category': stat_category
                                }
                                
                                for i, label in enumerate(stat_labels):
                                    if i < len(stats):
                                        player_stats[label] = stats[i]
                                
                                all_player_records.append({
                                    'player_id': str(player_id),
                                    'player_name': player_name,
                                    'team': team_name,
                                    'position': position,
                                    'game_id': game_id,
                                    'week': WEEK,
                                    'season': SEASON,
                                    'stat_category': stat_category,
                                    'stats': player_stats
                                })
                                game_player_count += 1
                    
                    print(f"✓ {game_player_count} records")
                else:
                    print("⚠️  unavailable")
                
                time.sleep(0.2)  # Rate limiting
                
            except Exception as e:
                print(f"❌ {e}")
                continue
        
        if all_player_records:
            # Convert to DataFrame
            rows = []
            for record in all_player_records:
                rows.append(
                    Row(
                        player_id=str(record['player_id']),
                        week=int(WEEK),
                        season=int(SEASON),
                        fantasy_points=0.0,
                        stats=json.dumps(record['stats']),
                        source='espn_public'
                    )
                )
            
            week_df = spark.createDataFrame(rows)
            
            # Aggregate multiple stat categories per player
            aggregated_df = week_df.groupBy("player_id", "week", "season", "source").agg(
                F.collect_list("stats").alias("all_stats")
            )
            
            def combine_stats(stats_list):
                import json
                combined = {}
                for stat_json in stats_list:
                    stat_dict = json.loads(stat_json)
                    stat_category = stat_dict.get('stat_category', 'unknown')
                    combined[stat_category] = stat_dict
                if stats_list:
                    first = json.loads(stats_list[0])
                    combined['player_name'] = first.get('player_name')
                    combined['team'] = first.get('team')
                    combined['position'] = first.get('position')
                    combined['game_id'] = first.get('game_id')
                    combined['game_name'] = first.get('game_name')
                return json.dumps(combined)
            
            combine_stats_udf = F.udf(combine_stats, StringType())
            
            bronze_df = aggregated_df.withColumn(
                "stats",
                combine_stats_udf(F.col("all_stats"))
            ).withColumn(
                "fantasy_points", F.lit(0.0)
            ).withColumn(
                "ingested_at", F.current_timestamp()
            ).select(
                "player_id", "week", "season", "fantasy_points", "stats", "source", "ingested_at"
            )
            
            # Write to bronze
            bronze_df.createOrReplaceTempView("latest_bronze_temp")
            spark.sql("""
                MERGE INTO main.fantasai.bronze_weekly_stats AS target
                USING latest_bronze_temp AS source
                ON target.player_id = source.player_id 
                    AND target.week = source.week 
                    AND target.season = source.season
                    AND target.source = source.source
                WHEN MATCHED THEN
                    UPDATE SET
                        target.fantasy_points = source.fantasy_points,
                        target.stats = source.stats,
                        target.ingested_at = source.ingested_at
                WHEN NOT MATCHED THEN
                    INSERT (player_id, week, season, fantasy_points, stats, source, ingested_at)
                    VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.source, source.ingested_at)
            """)
            
            # Write to silver
            silver_df = bronze_df.dropDuplicates(["player_id", "week", "season", "source"])
            silver_df.createOrReplaceTempView("latest_silver_temp")
            spark.sql("""
                MERGE INTO main.fantasai.silver_weekly_stats AS target
                USING latest_silver_temp AS source
                ON target.player_id = source.player_id 
                    AND target.week = source.week 
                    AND target.season = source.season
                    AND target.source = source.source
                WHEN MATCHED THEN
                    UPDATE SET
                        target.fantasy_points = source.fantasy_points,
                        target.stats = source.stats,
                        target.ingested_at = source.ingested_at
                WHEN NOT MATCHED THEN
                    INSERT (player_id, week, season, fantasy_points, stats, source, ingested_at)
                    VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.source, source.ingested_at)
            """)
            
            player_count = bronze_df.count()
            
            print(f"\n{'='*70}")
            print("FETCH COMPLETE")
            print(f"{'='*70}")
            print(f"\n✓ Season: {SEASON}")
            print(f"✓ Week: {WEEK}")
            print(f"✓ Unique players stored: {player_count}")
            print(f"✓ Total records: {len(all_player_records)}")
            print(f"\n📅 Data ingested at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        else:
            print("\n⚠️  No player data found")
    else:
        print(f"❌ Failed to fetch scoreboard: {response.status_code}")
        
except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()